# Gas–Oil Divergence to Alpha — U.S. LNG
Research-grade falsification pilot, 2026-10-19. **Not an investable backtest.**

## Hypotheses
H1: expensive WTI relative to Henry Hub on a rough energy-equivalent basis predicts future gas price increases. H2: the spread improves forecasting beyond momentum controls. H3: any improvement survives regime and lagged-placebo checks.


In [ ]:
from run_analysis import load_data,make_panel,evaluate,score,run
import pandas as pd
df=load_data()
print('Source rows:',len(df),'first:',df.date.min(),'last:',df.date.max())
display(df.tail())

## Point-in-time discipline
At forecast month t, all features use t−1 or earlier; y = 100·log[Gas(t+3)/Gas(t−1)]. Train rows u satisfy u+3 ≤ t−1. One-month source lag is an assumption, not verified historical vintage availability.


In [ ]:
panel=make_panel(df,3)
display(panel.head())
pred=evaluate(df,3)
display(pd.DataFrame(score(pred)).T)
assert len(pred)==242

## Robustness and negative controls
Expanding Ridge α=10 with train-only scaling and 96 minimum labels; historical-mean baseline, momentum controls, spread-enhanced model, and ablations. Test 1- and 6-month targets, 2016 LNG-era split, 2022 exclusion, and stale-spread 12/24/36-month placebos. Overlapping outcomes imply serial dependence.


In [ ]:
res=run(write=False)
for section in ('robustness','stale_spread_placebos'):
 print(section)
 display(pd.DataFrame({k:{'n':v['n'],'control_rmse':v['control']['rmse'],'full_rmse':v['full']['rmse'],'delta':v['delta_rmse_control_minus_full']} for k,v in res[section].items()}).T)
print('Moving-block bootstrap CI:',res['bootstrap_95_ci_delta_rmse'])

## Bias audit and next steps
Frozen EIA-derived data are **not** historical vintages. Spot-price changes are not tradable futures returns or equity alpha. Next: ALFRED/EIA release calendars, LNG export volumes, storage, weather, JKM/TTF netbacks, futures curve with rolls/fees, and company-level earnings revisions. See README.md and source register.


In [ ]:
run(write=True)
print('Wrote results/oos_predictions.csv and results/metrics.json')